# HW#5 Question Answering (Reading Comprehension) 
# Ximena Camacho
# CSC396 Fall 2025

In [ ]:
# Content
#  1) Initialization
#  2) Load and print training SQuAD json file
#  3) Preprocess the data
#  4) Configure and Train the model
#  5) Test the model with single entry
#  6) Load and print the dev SQuAD json file
#  7) Test small fraction of dev partition and print results
#  8) Final evaluation of complete dev partition
#  9) Error analysis of a few entries
# 10) Evaluation analysis -- Exact Match Score
# 11) References

Let's start with some initialization:

In [1]:
import random
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import pprint

# enable tqdm in pandas
tqdm.pandas()

# set to True to use the gpu (if there is one available)
use_gpu = True

# select device
device = torch.device('cuda' if use_gpu and torch.cuda.is_available() else 'cpu')
print(f'device: {device.type}')

# random seed
seed = 1234

# set random seed
if seed is not None:
    print(f'random seed: {seed}')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

device: cuda
random seed: 1234


In [ ]:
# Load more elements from transformers, as well as tool to load the dataset

In [2]:
import collections
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from transformers import pipeline

In [3]:
# Load the SQuAD JSON file
# Show content of training file
import json

with open('train-v1.1.json','r', encoding='utf-8') as f:
    data = json.load(f)

# Prepare lists to hold flattened data
ids = []
contexts = []
questions = []
answers = []

# Iterate through the JSON structure
for article in data['data']:
    for para in article['paragraphs']:
        context = para['context']
        for qa in para['qas']:
            ids.append(qa['id'])
            contexts.append(context)
            questions.append(qa['question'])
            # Handle multiple answers; however, I am taking the first one for simplicity
            answers.append(qa['answers'][0]['text'] if qa['answers'] else '')

# Create a Dataframe
df = pd.DataFrame({
    'id': ids,
    'context': contexts,
    'question': questions,
    'answer': answers
})

# If desired, the dataframe can be converted to csv format and saved to a file
#squadCsvFile = 'train-v1.1_converted.csv'
#df.to_csv(squadCsvFile, index=False)
#print("JSON to CSV conversion complete! Data saved to: " + squadCsvFile)

# Print training file
df


,id,context,question,answer
0,5733be284776f41900661182,"Architecturally, the school has a Catholic cha...",To whom did the Virgin Mary allegedly appear i...,Saint Bernadette Soubirous
1,5733be284776f4190066117f,"Architecturally, the school has a Catholic cha...",What is in front of the Notre Dame Main Building?,a copper statue of Christ
2,5733be284776f41900661180,"Architecturally, the school has a Catholic cha...",The Basilica of the Sacred heart at Notre Dame...,the Main Building
3,5733be284776f41900661181,"Architecturally, the school has a Catholic cha...",What is the Grotto at Notre Dame?,a Marian place of prayer and reflection
4,5733be284776f4190066117e,"Architecturally, the school has a Catholic cha...",What sits on top of the Main Building at Notre...,a golden statue of the Virgin Mary
...,...,...,...,...
87594,5735d259012e2f140011a09d,"Kathmandu Metropolitan City (KMC), in order to...",In what US state did Kathmandu first establish...,Oregon
87595,5735d259012e2f140011a09e,"Kathmandu Metropolitan City (KMC), in order to...",What was Yangon previously known as?,Rangoon
87596,5735d259012e2f140011a09f,"Kathmandu Metropolitan City (KMC), in order to...",With what Belorussian city does Kathmandu have...,Minsk
87597,5735d259012e2f140011a0a0,"Kathmandu Metropolitan City (KMC), in order to...",In what year did Kathmandu create its initial ...,1975


In [4]:
# Load SQuAD dataset
raw_datasets = load_dataset("squad")

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
# Show dataset structure
raw_datasets

Some weights of DistilBertForQuestionAnswering were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [5]:
# Preprocess the data
max_length = 384
stride = 128

def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end token indices
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        start_token = idx
        while idx < len(sequence_ids) and sequence_ids[idx] != None:
            idx += 1
        end_token = idx - 1

        # Adapt answer start/end tokens to the span
        if offset[start_token][0] > start_char or offset[end_token][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            while start_token < len(offset) and offset[start_token][0] <= start_char:
                start_token += 1
            start_positions.append(start_token - 1)
            while end_token >= 0 and offset[end_token][1] >= end_char:
                end_token -= 1
            end_positions.append(end_token + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

# Map the tokenized train datasets
tokenized_train_datasets = raw_datasets["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)


In [6]:
# Configure and Train the Model
# I am training the model following a similar approach used in some of the notebooks
# covered in class this semester, CSC396 Fall 2025
# I am also using some suggested values from external resources
args = TrainingArguments(
    output_dir="qa_model",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    weight_decay=0.01,
    push_to_hub=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train_datasets,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
)

print("Starting training!!!")
trainer.train()
print("Training finished!!!.")
trainer.save_model("my_qa_model")


/tmp/ipykernel_412642/1456673900.py:15: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Starting training!!!


Step,Training Loss
500,2.935200
1000,1.767100
1500,1.591900
2000,1.515000
2500,1.450900
3000,1.336900
3500,1.326000
4000,1.309200
4500,1.235900
5000,1.233800


Training finished!!!.


In [7]:
# A decreasing training loss indicates that the model is learning and generalizing well.
# Another parameter to observe would be the validation loss, but I do not have it readily available

In [8]:
# Test the model
# This is an example to test whether the model is working

# Load the model I just trained using pipeline for easy inference
qa_pipeline = pipeline("question-answering", model="my_qa_model", tokenizer=tokenizer)

context = """
The Amazon rainforest is a moist broadleaf tropical rainforest in the Amazon biome that covers 
most of the Amazon basin of South America. This basin encompasses 7 million square kilometers 
(2.7 million square miles), of which 5.5 million square kilometers (2.1 million square miles) 
are covered by the rainforest. This region includes territory belonging to nine nations.
"""
question = "Which continent is the Amazon rainforest in?"

result = qa_pipeline(question=question, context=context)
print(f"\nQuestion: {question}")
print(f"Predicted Answer: {result['answer']}")
print(f"Confidence Score: {result['score']:.4f}")

Device set to use cuda:0



Question: Which continent is the Amazon rainforest in?
Predicted Answer: South America
Confidence Score: 0.9083


In [9]:
# Given the context, the answer is correct, so the model seems to be working!

In [10]:
# Load the dev SQuAD JSON file
# Show content of dev file to be used for evaluation
import json

with open('dev-v1.1.json','r', encoding='utf-8') as f:
    data = json.load(f)

# Prepare lists to hold flattened data
ids = []
contexts = []
questions = []
answers = []

# Iterate through the JSON structure
for article in data['data']:
    for para in article['paragraphs']:
        context = para['context']
        for qa in para['qas']:
            ids.append(qa['id'])
            contexts.append(context)
            questions.append(qa['question'])
            # Handle multiple answers, however, I am taking the first one for simplicity
            answers.append(qa['answers'][0]['text'] if qa['answers'] else '')

# Create a Dataframe
df = pd.DataFrame({
    'id': ids,
    'context': contexts,
    'question': questions,
    'answer': answers
})

# If desired, the dataframe can be converted to csv format and saved to a file
#squadCsvFile = 'dev-v1.1_converted.csv'
#df.to_csv(squadCsvFile, index=False)
#print("JSON to CSV conversion complete! Data saved to: " + squadCsvFile)
# Print DEV file
df


,id,context,question,answer
0,56be4db0acb8001400a502ec,Super Bowl 50 was an American football game to...,Which NFL team represented the AFC at Super Bo...,Denver Broncos
1,56be4db0acb8001400a502ed,Super Bowl 50 was an American football game to...,Which NFL team represented the NFC at Super Bo...,Carolina Panthers
2,56be4db0acb8001400a502ee,Super Bowl 50 was an American football game to...,Where did Super Bowl 50 take place?,"Santa Clara, California"
3,56be4db0acb8001400a502ef,Super Bowl 50 was an American football game to...,Which NFL team won Super Bowl 50?,Denver Broncos
4,56be4db0acb8001400a502f0,Super Bowl 50 was an American football game to...,What color was used to emphasize the 50th anni...,gold
...,...,...,...,...
10565,5737aafd1c456719005744fb,"The pound-force has a metric counterpart, less...",What is the metric term less used than the New...,kilogram-force
10566,5737aafd1c456719005744fc,"The pound-force has a metric counterpart, less...",What is the kilogram-force sometimes reffered ...,kilopond
10567,5737aafd1c456719005744fd,"The pound-force has a metric counterpart, less...",What is a very seldom used unit of mass in the...,slug
10568,5737aafd1c456719005744fe,"The pound-force has a metric counterpart, less...",What seldom used term of a unit of force equal...,kip


In [11]:
# Below is the ouptut of the first element in the dev partition
print(f"\nContext: {df["context"][0]}")
print(f"\nQuestion: {df["question"][0]}")
print(f"\nAnswer: {df["answer"][0]}")
len_dev_file = len(df["context"])
print(f"\nThe length of the dev partition file is: {len_dev_file}")



Context: Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.

Question: Which NFL team represented the AFC at Super Bowl 50?

Answer: Denver Broncos

The length of the dev partition file is: 10570


In [12]:
# Evaluate small fraction of dev file
# To evaluate how good my model is, I entered some of the questions found in the dev file.
# I compared the predicted answer against the answer given in the dev partition, which is what I
# am assuming is implied with "Exact Match".

# In this cell I am evaluating a few entries to show the output of my approach, including some
# incorrect predictions

# Load the model I just trained using pipeline for easy inference
qa_pipeline = pipeline("question-answering", model="my_qa_model", tokenizer=tokenizer)

# Entries to evaluate
elem_to_eval = 15 # Set it initially to a low value for testing purposes
# Ensure the number of entries is not larger than the actual length of file. If so, set the number to the
# length of file.
if elem_to_eval > len_dev_file:
    print(f"\nNumber of lines cropped to max length of file")
    elem_to_eval = len_dev_file
    print(f"Evaluating {elem_to_eval} entries")

correct_answers_count = 0
incorrect_answers_index = []

print("Start evaluation of small fraction of dev partition\n")

for index in range(elem_to_eval):
    context = df["context"][index]
    question = df["question"][index]
    answer = df["answer"][index]
    result = qa_pipeline(question=question, context=context)
    predicted_answer = result['answer']

    print(question)
    # Correct answer
    print(answer)
    # Predicted answer
    print(predicted_answer)

    if predicted_answer == answer:
        correct_answers_count += 1
    else:
        # Save indexes of incorrect answers for error analysis
        incorrect_answers_index.append(index)

correct_answers_perc = (correct_answers_count / elem_to_eval) * 100

print(f"\nThe percentage of correct answers using small fraction of the dev partition is: {correct_answers_perc:.2f}%")

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Start evaluation of small fraction of dev partition

Which NFL team represented the AFC at Super Bowl 50?
Denver Broncos
Denver Broncos
Which NFL team represented the NFC at Super Bowl 50?
Carolina Panthers
Carolina Panthers
Where did Super Bowl 50 take place?
Santa Clara, California
Levi's Stadium
Which NFL team won Super Bowl 50?
Denver Broncos
Denver Broncos
What color was used to emphasize the 50th anniversary of the Super Bowl?
gold
gold
What was the theme of Super Bowl 50?
"golden anniversary"
"golden anniversary"
What day was the game played on?
February 7, 2016
February 7, 2016
What is the AFC short for?
American Football Conference
American Football Conference
What was the theme of Super Bowl 50?
"golden anniversary"
"golden anniversary"
What does AFC stand for?
American Football Conference
American Football Conference
What day was the Super Bowl played on?
February 7, 2016
February 7, 2016
Who won Super Bowl 50?
Denver Broncos
Denver Broncos
What venue did Super Bowl 50 take 

In [13]:
# Final evaluation of complete dev partition file
# To evaluate how good my model is, I entered some of the questions found in the dev file.
# I compared the predicted answer against the answer given in the dev partition, which is what I am
# assuming is implied with "Exact Match".

# Load the model I just trained using pipeline for easy inference
qa_pipeline = pipeline("question-answering", model="my_qa_model", tokenizer=tokenizer)

# Entries to evaluate
elem_to_eval = 10570 #Length of dev partition file
# Ensure the number of entries is not larger than the actual length of the file. If so, set the number to
# the length of the file.
if elem_to_eval > len_dev_file:
    print(f"\nNumber of lines cropped to max length of file")
    elem_to_eval = len_dev_file
    print(f"Evaluating {elem_to_eval} entries")

correct_answers_count = 0
incorrect_answers_index = []

print("Start evaluation of dev partition")

for index in range(elem_to_eval):
    context = df["context"][index]
    question = df["question"][index]
    answer = df["answer"][index]
    result = qa_pipeline(question=question, context=context)
    predicted_answer = result['answer']

    # print(question)
    # # Correct answer
    # print(answer)
    # # Predicted answer
    # print(predicted_answer)

    if predicted_answer == answer:
        correct_answers_count += 1
    else:
        # Save indexes of incorrect answers for error analysis
        incorrect_answers_index.append(index)

correct_answers_perc = (correct_answers_count / elem_to_eval) * 100

print(f"\nThe percentage of correct answers using the dev partition is: {correct_answers_perc:.2f}%")

Device set to use cuda:0


Start evaluation of dev partition

The percentage of correct answers using the dev partition is: 57.71%


In [14]:
print(f"\nIncorrect answers: {len(incorrect_answers_index)}")


Incorrect answers: 4470


In [15]:
# Error analysis

# Here I am showing a few entries where the model predicted incorrectly.

# Load the model I just trained using pipeline for easy inference
qa_pipeline = pipeline("question-answering", model="my_qa_model", tokenizer=tokenizer)

for inc_index in range(5, 10):
    index = incorrect_answers_index[inc_index]
    context = df["context"][index]
    question = df["question"][index]
    answer = df["answer"][index]
    result = qa_pipeline(question=question, context=context)
    predicted_answer = result['answer']

    print("\ncontext: ")
    print(context)
    print(question)
    # Correct answer
    print(answer)
    # Predicted answer
    print(predicted_answer)


Device set to use cuda:0



context: 
Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.
The name of the NFL championship game is?
Super Bowl
Super Bowl 50

context: 
The Panthers finished the regular season with a 15–1 record, and quarterback Cam Newton was named the NFL Most Valuable Player (MVP). T

In [ ]:
# In the output above, one can see that the model clearly made some inaccurate predictions, for example, 
# (New England Patriots vs Arizona Cardinals), while in other cases it is just semantics, 
# 8 vs eight, as well as Super Bowl vs Super Bowl 50.
# Not shown in these examples, but the model also had difficulties determining locations (e.g; Stadium
# vs City)

In [ ]:
# Evaluation Analysis
# Paper Exact Match Score: 40.0%
# My implementation Exact Match Score: 57.71%

# Potential causes for discrepancy with respect to the Exact Match score:

# 1) I am not sure whether the approach I used to compute the Exact Match Score is the most
# appropriate, therefore, the score reported in the paper could not refer exactly to what I am computing.
# For time purposes, I am only using the first truth answer, from the three provided, to compute my score.

# 2) I also had some difficulties finding the original dataset (train & dev) from the website provided
# in the assigment. Version 2.0 was readily available, but 1.1 was not. So, I searched other websites and 
# found what seemed to be a reduced version of them. I am assuming that the files I used are in
# fact a reduced version of the original and contain valid information (data has not been modified).

# 3) It is also possible that by using the complete dataset versions, the score could drop and get 
# closer to the published result, instead of there being a 17.71% difference.

# 4) Finally, the paper was also published in 2016 and some of the difference in the score values could
# be due to currently better developed tools. In the conclusion of the paper, the authors mention that
# since the release of their dataset, they have seen considerable interest in building models on the
# dataset. As a result, the gap between their logistic regression model and human performance has more
# than halved, which makes my model performance even more comparable to the score results.


In [ ]:
# References
# 1) Paper: "SQuAD: 100,000+ Questions for Machine Comprehension of Text," P. Rajpurkar, 
#    J. Zhang, K. Lopyrev, and P. Liang, Computer Science Department, Standford University, Oct 2016
# 2) Rajipurkar, Pranav. "Squad2.0." The Stanford Question Answering Dataset, rajpurkar.github.io/SQuAD-explorer/.
# 3) CSC396 Fall 2025 notes
# 4) CSC396 Fall 2025 jupyter notebooks
# 5) Other external sources
